# 01 · Study Design

Compare run-first with predict-first. Each condition provides one practice run per round and is followed by an unaided test-design task. Review the protocol before any participant work. Upload the project ZIP when prompted.

In [ ]:
import io
import json
import sys
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

root = next((p for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "check-before-you-run")
             if (p / "src" / "study.py").is_file()), None)
if root is None:
    from google.colab import files
    uploads = files.upload()
    if len(uploads) != 1:
        raise ValueError("Upload the original project ZIP.")
    with ZipFile(io.BytesIO(next(iter(uploads.values())))) as archive:
        paths = [PurePosixPath(name) for name in archive.namelist()]
        if not paths or any(path.is_absolute() or ".." in path.parts for path in paths):
            raise ValueError("Invalid project archive.")
        archive.extractall(Path.cwd())
    root = Path.cwd() / "check-before-you-run"
    if not (root / "src" / "study.py").is_file():
        raise ValueError("Use the original check-before-you-run.zip.")
root = root.resolve()
sys.path.insert(0, str(root))
from src import study

print("Project ready")

## Verify the eight sequences

Each participant sees each topic once and completes two rounds per condition. Each topic appears once per condition at every position across the full assignment.

In [ ]:
from collections import Counter
from random import Random

sequences = {index: study.schedule(index) for index in range(8)}
counts = Counter(
    (position, task, condition)
    for sequence in sequences.values()
    for position, (task, condition) in enumerate(sequence, 1)
)
assert len(counts) == 32 and set(counts.values()) == {1}
assert all(Counter(condition for _, condition in sequence) == {"A": 2, "B": 2}
           for sequence in sequences.values())
slots = list(sequences)
Random(37).shuffle(slots)
for index, slot in enumerate(slots, 1):
    print(f"P{index:02d} | sequence {slot} | {sequences[slot]}")